In [ ]:
# AI CAPEX and GDP — Simplified v2
# 1. Assumptions
#
# This notebook estimates nominal net AI GDP flow and, for actual quarters,
# an approximate contribution to q/q annualized real GDP growth:
#     net AI GDP flow = US AI investment recognized in GDP - associated imports
#
# It does not project GDP beyond the latest actual quarter and does not include
# an export adjustment. Real contributions are additive chain-dollar approximations.

from pathlib import Path
import json
import re
import urllib.request
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

# -----------------------------
# Files and model dates
# -----------------------------
INPUT_FILE = "capex_final_data.csv"
OUTPUT_FILE = "results_simplified_v2.xlsx"
LAST_ACTUAL_QUARTER = pd.Period("2026Q2", freq="Q")
MODEL_START = pd.Period("2022Q1", freq="Q")
ANALYSIS_END = pd.Period("2028Q4", freq="Q")
TAIL_END = pd.Period("2029Q3", freq="Q")
REAL_CONTRIBUTION_END = LAST_ACTUAL_QUARTER
EXPORT_OUTPUTS = True

# -----------------------------
# Core methodological assumptions
# -----------------------------
BASELINE_YEAR = 2022
POSITIVE_EXCESS_ONLY = True       # All positive company-quarter excess is treated as AI-related.
INCLUDE_EXPORTS = False           # Net flow subtracts imports only; it is not a full net-exports estimate.
USE_REPORTED_CAPEX_ONLY = True
REQUIRE_SAME_QUARTER_2022_BASELINE = True
ZERO_BASELINE_FOR_POST_2022_ENTRANTS = False
QOQ_ANNUALIZATION_EXPONENT = 4
EXPECTED_2026Q2_REAL_GDP_GROWTH_SAAR = 1.5  # Rounded BEA headline; diagnostic only.

# Only CAPEX values present in the source CSV are used. Missing actuals and estimates
# remain missing: there is no interpolation, run-rate fill, or seasonal adjustment.

# The counterfactual for each reported company-quarter is that company's reported
# CAPEX in the same calendar quarter of 2022. If that exact 2022 observation is
# unavailable, the observation is excluded from AI excess rather than assigned a zero
# baseline. Negative excess does not offset positive excess.

# -----------------------------
# US geographic shares
# -----------------------------
US_SHARE_DEFAULT = 0.70
US_SHARE_UTILITY = 1.00
COMPANY_US_SHARE_OVERRIDES = {}

# -----------------------------
# Capital-profile assignment
# -----------------------------
PROFILE_OVERRIDES = {
    "Microsoft Corp": "hyperscaler_cloud",
    "Amazon.com Inc": "hyperscaler_cloud",
    "Alphabet Inc": "hyperscaler_cloud",
    "Meta Platforms Inc": "hyperscaler_cloud",
    "Oracle Corp": "hyperscaler_cloud",
    "CoreWeave Inc": "hyperscaler_cloud",
    "Applied Digital Corp": "hyperscaler_cloud",
    "DigitalOcean Holdings Inc": "hyperscaler_cloud",
    "Cloudflare Inc": "hyperscaler_cloud",
    "Tesla Inc": "tesla_automotive",
}

INDUSTRY_PROFILE = {
    "Software & Services": "software_services",
    "Technology Hardware & Equipment": "tech_hardware",
    "Utilities": "utilities",
    "Media & Entertainment": "hyperscaler_cloud",
    "Automobiles & Components": "tesla_automotive",
    "Consumer Discretionary Distribution & Retail": "hyperscaler_cloud",
}

# Shares include land, which is excluded rather than reallocated.
CATEGORY_SPLITS = pd.DataFrame.from_dict(
    {
        "hyperscaler_cloud": {
            "dc_structures": 0.25, "power_structures": 0.00,
            "it_equipment": 0.55, "power_cooling_equipment": 0.10,
            "other_pfi": 0.05, "land_excluded": 0.05,
        },
        "software_services": {
            "dc_structures": 0.10, "power_structures": 0.00,
            "it_equipment": 0.50, "power_cooling_equipment": 0.15,
            "other_pfi": 0.20, "land_excluded": 0.05,
        },
        "tech_hardware": {
            "dc_structures": 0.05, "power_structures": 0.00,
            "it_equipment": 0.25, "power_cooling_equipment": 0.15,
            "other_pfi": 0.50, "land_excluded": 0.05,
        },
        "utilities": {
            "dc_structures": 0.00, "power_structures": 0.55,
            "it_equipment": 0.00, "power_cooling_equipment": 0.35,
            "other_pfi": 0.05, "land_excluded": 0.05,
        },
        "tesla_automotive": {
            "dc_structures": 0.00, "power_structures": 0.00,
            "it_equipment": 0.05, "power_cooling_equipment": 0.10,
            "other_pfi": 0.80, "land_excluded": 0.05,
        },
    },
    orient="index",
)

# Recognition weights from the company CAPEX quarter onward.
LAG_WEIGHTS = {
    "dc_structures": [0.30, 0.30, 0.20, 0.20],
    "power_structures": [0.25, 0.25, 0.25, 0.25],
    "it_equipment": [0.60, 0.30, 0.10],
    "power_cooling_equipment": [0.50, 0.30, 0.20],
    "other_pfi": [0.70, 0.20, 0.10],
}

# Single central import-share assumption. Imports use the same recognition timing
# as the associated investment. Structures are assumed to have zero import content.
IMPORT_SHARES = {
    "dc_structures": 0.00,
    "power_structures": 0.00,
    "it_equipment": 0.60,
    "power_cooling_equipment": 0.30,
    "other_pfi": 0.25,
}

# -----------------------------
# Actual real-GDP contribution assumptions
# -----------------------------
# Investment and imports are deflated separately. IT imports use the BEA computer-
# import price index. Other imports use the matching domestic asset price index as
# a proxy. The power/cooling deflator is a 50/50 average of communication and
# electrical-equipment price indexes. The data-center index is ratio-spliced to
# the office-structures price index before the data-center series begins.
POWER_COOLING_PRICE_WEIGHTS = {
    "communication_equipment_price": 0.50,
    "electrical_equipment_price": 0.50,
}

BEA_BULK_FILES = {
    "section1": ("bea_section1.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section1All_xls.xlsx"),
    "section4": ("bea_section4.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section4All_xls.xlsx"),
    "section5": ("bea_section5.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section5All_xls.xlsx"),
    "underlying5": ("bea_underlying_section5.xlsx", "https://apps.bea.gov/national/Release/XLS/Underlying/Section5All_xls.xlsx"),
}

REAL_SERIES_REGISTRY = {
    "real_gdp": dict(bulk="section1", sheet="T10106-Q", code="A191RX", label="Gross domestic product", scale=0.001),
    "computers_price": dict(bulk="section5", sheet="T50304-Q", code="B935RG", label="Computers and peripheral equipment", scale=1.0),
    "other_equipment_price": dict(bulk="section5", sheet="T50304-Q", code="A862RG", label="Other equipment", scale=1.0),
    "office_price": dict(bulk="underlying5", sheet="U50404-Q", code="W031RG", label="Office", scale=1.0),
    "dc_structures_price": dict(bulk="underlying5", sheet="U50404-Q", code="IA001282", label="Data centers", scale=1.0),
    "power_structures_price": dict(bulk="underlying5", sheet="U50404-Q", code="W028RG", label="Power", scale=1.0),
    "communication_equipment_price": dict(bulk="underlying5", sheet="U50504-Q", code="B275RG", label="Communication equipment", scale=1.0),
    "electrical_equipment_price": dict(bulk="underlying5", sheet="U50504-Q", code="B276RG", label="Electrical transmission", scale=1.0),
    "computer_imports_price": dict(bulk="section4", sheet="T40204B-Q", code="B852RG", label="Computers, peripherals, and parts", scale=1.0),
}

assert np.allclose(CATEGORY_SPLITS.sum(axis=1), 1.0)
assert all(np.isclose(sum(weights), 1.0) for weights in LAG_WEIGHTS.values())

ROOT = Path.cwd()
CSV_PATH = ROOT / INPUT_FILE
CACHE_DIR = ROOT / "data_cache"
OUTPUT_DIR = ROOT / "output"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print("Central outputs: nominal net AI GDP flow and actual-quarter real GDP growth contribution")
display(CATEGORY_SPLITS.map(lambda value: f"{value:.0%}"))
display(pd.Series(IMPORT_SHARES, name="import_share").map(lambda value: f"{value:.0%}"))


# 2. Company CAPEX

Load and audit the source file, classify reported observations as actual or estimated, and use only the CAPEX values present in the file. Missing observations remain missing.


In [ ]:
# Load and audit the company CAPEX data
REQUIRED_COLUMNS = ["Quarter_End_Date", "Company_Name", "GICS_Industry_Group", "CAPEX"]
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {CSV_PATH}")

capex_raw = pd.read_csv(CSV_PATH)
missing_columns = set(REQUIRED_COLUMNS).difference(capex_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

capex_raw = capex_raw[REQUIRED_COLUMNS].copy()
capex_raw["Quarter_End_Date"] = pd.to_datetime(capex_raw["Quarter_End_Date"], errors="raise")
capex_raw["quarter"] = capex_raw["Quarter_End_Date"].dt.to_period("Q")
capex_raw["CAPEX"] = pd.to_numeric(capex_raw["CAPEX"], errors="coerce")
capex_raw["Company_Name"] = capex_raw["Company_Name"].astype(str).str.strip()
capex_raw["GICS_Industry_Group"] = capex_raw["GICS_Industry_Group"].astype(str).str.strip()

duplicate_count = int(capex_raw.duplicated(["Company_Name", "quarter"]).sum())
blank_industry_rows = int(capex_raw["GICS_Industry_Group"].replace("", np.nan).isna().sum())
industry_conflicts = int((capex_raw.groupby("Company_Name")["GICS_Industry_Group"].nunique(dropna=True) != 1).sum())
total_2024 = capex_raw.loc[capex_raw["quarter"].dt.year.eq(2024), "CAPEX"].sum(min_count=1)

input_audit = pd.DataFrame({
    "metric": [
        "rows", "companies", "duplicate company-quarters", "blank industry rows",
        "companies with conflicting industries", "missing CAPEX values", "2024 CAPEX ($bn)"
    ],
    "value": [
        len(capex_raw), capex_raw["Company_Name"].nunique(), duplicate_count,
        blank_industry_rows, industry_conflicts, int(capex_raw["CAPEX"].isna().sum()), total_2024
    ],
})
display(input_audit)

if duplicate_count or blank_industry_rows or industry_conflicts:
    raise ValueError("Company-quarter or industry-group validation failed.")
if abs(total_2024 - 490.9) > 1.0:
    warnings.warn(f"2024 CAPEX is ${total_2024:,.1f}bn; expected approximately $490.9bn.")


In [ ]:
# Use reported company CAPEX only — no interpolation or imputation
panel = capex_raw.rename(columns={"CAPEX": "capex_reported"}).copy()
panel["year"] = panel["quarter"].dt.year
panel["quarter_num"] = panel["quarter"].dt.quarter
panel["data_status"] = np.select(
    [
        panel["quarter"].le(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].gt(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].le(LAST_ACTUAL_QUARTER),
    ],
    ["actual", "reported_estimate", "missing_actual"],
    default="missing_estimate",
)

company_meta = panel.groupby("Company_Name", as_index=False).agg(
    GICS_Industry_Group=("GICS_Industry_Group", "first")
)

reported_coverage = panel.groupby("quarter").agg(
    reported_companies=("capex_reported", lambda values: int(values.notna().sum())),
    reported_capex_bn=("capex_reported", lambda values: values.sum(min_count=1)),
    actual_companies=("data_status", lambda values: int(values.eq("actual").sum())),
    reported_estimate_companies=("data_status", lambda values: int(values.eq("reported_estimate").sum())),
)
display(reported_coverage.loc[pd.Period("2025Q1", freq="Q"):ANALYSIS_END])


# 3. AI excess

For every reported company-quarter, compare CAPEX with the same company's CAPEX in the same calendar quarter of 2022. Observations without that exact 2022 comparison are excluded. Positive excess is treated as AI-related CAPEX.


In [ ]:
# Exact same-company, same-calendar-quarter 2022 baseline
baseline_lookup = panel.loc[
    panel["year"].eq(BASELINE_YEAR) & panel["capex_reported"].notna(),
    ["Company_Name", "quarter_num", "capex_reported"],
].rename(columns={"capex_reported": "baseline_same_quarter_2022_bn"})

panel = panel.merge(baseline_lookup, on=["Company_Name", "quarter_num"], how="left")
panel["baseline_source"] = np.where(
    panel["baseline_same_quarter_2022_bn"].notna(),
    "same_quarter_2022",
    "missing_same_quarter_2022",
)
panel["baseline_flat_bn"] = panel["baseline_same_quarter_2022_bn"].where(
    panel["capex_reported"].notna()
)
panel["raw_excess_bn"] = panel["capex_reported"] - panel["baseline_flat_bn"]
panel["gross_ai_capex_bn"] = (
    panel["raw_excess_bn"].clip(lower=0)
    if POSITIVE_EXCESS_ONLY
    else panel["raw_excess_bn"]
)
panel["baseline_eligible"] = (
    panel["capex_reported"].notna() & panel["baseline_flat_bn"].notna()
)
panel["baseline_eligible_capex_bn"] = panel["capex_reported"].where(panel["baseline_eligible"])
panel["capex_excluded_missing_baseline_bn"] = panel["capex_reported"].where(
    panel["capex_reported"].notna() & panel["baseline_flat_bn"].isna()
)

baseline_stats = baseline_lookup.groupby("Company_Name").agg(
    valid_2022_quarters=("baseline_same_quarter_2022_bn", "count"),
    baseline_annual_2022_bn=("baseline_same_quarter_2022_bn", lambda values: values.sum(min_count=1)),
).reset_index()
baseline_audit = company_meta.merge(baseline_stats, on="Company_Name", how="left")
baseline_audit["valid_2022_quarters"] = baseline_audit["valid_2022_quarters"].fillna(0).astype(int)
baseline_audit["baseline_status"] = np.select(
    [
        baseline_audit["valid_2022_quarters"].eq(4),
        baseline_audit["valid_2022_quarters"].gt(0),
    ],
    ["complete_2022", "partial_2022"],
    default="missing_2022",
)

coverage_audit = panel.groupby("quarter").agg(
    reported_companies=("capex_reported", lambda values: int(values.notna().sum())),
    reported_capex_bn=("capex_reported", lambda values: values.sum(min_count=1)),
    baseline_eligible_companies=("baseline_eligible", "sum"),
    baseline_eligible_capex_bn=("baseline_eligible_capex_bn", lambda values: values.sum(min_count=1)),
    excluded_missing_baseline_capex_bn=("capex_excluded_missing_baseline_bn", lambda values: values.sum(min_count=1)),
    actual_companies=("data_status", lambda values: int(values.eq("actual").sum())),
    reported_estimate_companies=("data_status", lambda values: int(values.eq("reported_estimate").sum())),
)
coverage_audit["baseline_coverage_share"] = coverage_audit["baseline_eligible_capex_bn"].div(
    coverage_audit["reported_capex_bn"].replace(0, np.nan)
)

display(baseline_audit["baseline_status"].value_counts().rename_axis("baseline_status").to_frame("companies"))
display(coverage_audit.loc[pd.Period("2025Q1", freq="Q"):ANALYSIS_END])


# 4. US investment

Apply the assumed US geographic share, assign each company to a capital profile, distribute AI excess across the modeled asset categories, and exclude land.


In [ ]:
# Apply US shares and distribute US AI investment across asset categories
panel["capital_profile"] = panel["Company_Name"].map(PROFILE_OVERRIDES)
panel["capital_profile"] = panel["capital_profile"].fillna(
    panel["GICS_Industry_Group"].map(INDUSTRY_PROFILE)
)
if panel["capital_profile"].isna().any():
    missing_profiles = sorted(panel.loc[panel["capital_profile"].isna(), "GICS_Industry_Group"].unique())
    raise ValueError(f"Missing capital-profile mappings for: {missing_profiles}")

panel["us_share"] = np.where(panel["capital_profile"].eq("utilities"), US_SHARE_UTILITY, US_SHARE_DEFAULT)
panel["us_share"] = panel["Company_Name"].map(COMPANY_US_SHARE_OVERRIDES).fillna(panel["us_share"])
panel["us_ai_capex_source_bn"] = panel["gross_ai_capex_bn"] * panel["us_share"]

model_panel = panel.loc[panel["quarter"].between(MODEL_START, ANALYSIS_END)].copy()
allocation_blocks = []
for category in CATEGORY_SPLITS.columns:
    block = model_panel[[
        "Company_Name", "GICS_Industry_Group", "capital_profile", "quarter", "us_ai_capex_source_bn"
    ]].copy()
    block["category"] = category
    block["category_share"] = block["capital_profile"].map(CATEGORY_SPLITS[category])
    block["source_amount_bn"] = block["us_ai_capex_source_bn"] * block["category_share"]
    allocation_blocks.append(block)
allocated = pd.concat(allocation_blocks, ignore_index=True)

land_excluded = allocated.loc[allocated["category"].eq("land_excluded")].copy()
us_investment_by_asset = allocated.loc[
    ~allocated["category"].eq("land_excluded")
].groupby("category")["source_amount_bn"].sum().to_frame("us_ai_investment_source_bn")
display(us_investment_by_asset)


# 5. GDP recognition

Apply the asset-specific recognition weights to bridge company cash CAPEX timing to the quarter in which investment is assumed to enter measured private fixed investment.


In [ ]:
# Apply asset-specific timing to determine when investment is recognized in GDP
timed_blocks = []
for category, weights in LAG_WEIGHTS.items():
    category_source = allocated.loc[allocated["category"].eq(category)].copy()
    for lag, weight in enumerate(weights):
        recognized = category_source.copy()
        recognized["source_quarter"] = recognized["quarter"]
        recognized["recorded_quarter"] = recognized["quarter"] + lag
        recognized["lag"] = lag
        recognized["lag_weight"] = weight
        recognized["gross_ai_investment_bn"] = recognized["source_amount_bn"] * weight
        timed_blocks.append(recognized)

asset_detail = pd.concat(timed_blocks, ignore_index=True)
asset_detail = asset_detail.loc[asset_detail["recorded_quarter"].le(TAIL_END)].copy()
display(asset_detail.groupby("category")["gross_ai_investment_bn"].sum().to_frame())


# 6. Imports

Apply the asset-specific import shares to recognized investment. Nominal net AI GDP flow equals recognized US AI investment minus associated imports.


In [ ]:
# Estimate associated imports and the nominal net AI GDP flow
asset_detail["import_share"] = asset_detail["category"].map(IMPORT_SHARES)
asset_detail["associated_imports_bn"] = asset_detail["gross_ai_investment_bn"] * asset_detail["import_share"]
asset_detail["net_ai_gdp_flow_bn"] = asset_detail["gross_ai_investment_bn"] - asset_detail["associated_imports_bn"]

display(asset_detail.groupby("category")[[
    "gross_ai_investment_bn", "associated_imports_bn", "net_ai_gdp_flow_bn"
]].sum())


In [ ]:
# Summarize nominal US investment, imports, and net GDP flows
quarterly_flows = asset_detail.groupby("recorded_quarter").agg(
    gross_ai_investment_bn=("gross_ai_investment_bn", "sum"),
    associated_imports_bn=("associated_imports_bn", "sum"),
    net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", "sum"),
).reindex(pd.period_range(MODEL_START, TAIL_END, freq="Q"), fill_value=0.0)
quarterly_flows.index.name = "quarter"

source_quarterly = model_panel.groupby("quarter").agg(
    company_capex_bn=("capex_reported", "sum"),
    flat_2022_baseline_bn=("baseline_flat_bn", "sum"),
    gross_ai_capex_bn=("gross_ai_capex_bn", "sum"),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", "sum"),
)
quarterly_results = source_quarterly.join(quarterly_flows, how="outer")
quarterly_results["data_status"] = np.where(
    quarterly_results.index <= LAST_ACTUAL_QUARTER, "Actual", "Estimate"
)

annual_results = quarterly_results.groupby(quarterly_results.index.year).agg(
    company_capex_bn=("company_capex_bn", "sum"),
    flat_2022_baseline_bn=("flat_2022_baseline_bn", "sum"),
    gross_ai_capex_bn=("gross_ai_capex_bn", "sum"),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", "sum"),
    gross_ai_investment_bn=("gross_ai_investment_bn", "sum"),
    associated_imports_bn=("associated_imports_bn", "sum"),
    net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", "sum"),
)
annual_results.index.name = "year"
annual_results["data_status"] = np.select(
    [annual_results.index <= 2025, annual_results.index == 2026],
    ["Actual", "Actual through Q2 / estimate thereafter"],
    default="Estimate",
)

industry_breakdown = model_panel.groupby(
    [model_panel["quarter"].dt.year, "GICS_Industry_Group"]
).agg(
    gross_ai_capex_bn=("gross_ai_capex_bn", "sum"),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", "sum"),
).reset_index().rename(columns={"quarter": "year"})

asset_breakdown = asset_detail.groupby(
    [asset_detail["recorded_quarter"].dt.year, "category"]
).agg(
    gross_ai_investment_bn=("gross_ai_investment_bn", "sum"),
    associated_imports_bn=("associated_imports_bn", "sum"),
    net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", "sum"),
).reset_index().rename(columns={"recorded_quarter": "year"})

display(annual_results.loc[2023:2028])
display(quarterly_results.loc[pd.Period("2025Q1", freq="Q"):ANALYSIS_END].tail(12))


# 7. Estimated growth contributions

Deflate gross investment and imports separately, convert the quarterly flows to SAAR, and compare actual BEA real GDP growth with a constructed no-AI counterfactual. The result is an approximate contribution to q/q annualized real GDP growth because chain dollars are not strictly additive and company CAPEX is not seasonally adjusted.


In [ ]:
# Load the minimum public BEA data needed for the real-growth estimate
def load_bea_bulk_series(name):
    entry = REAL_SERIES_REGISTRY[name]
    filename, url = BEA_BULK_FILES[entry["bulk"]]
    path = CACHE_DIR / filename
    if not path.exists():
        print(f"Downloading {filename} ...")
        urllib.request.urlretrieve(url, path)

    frame = pd.read_excel(path, sheet_name=entry["sheet"], header=7)
    frame = frame.rename(columns={
        frame.columns[0]: "Line", frame.columns[1]: "Description", frame.columns[2]: "SeriesCode"
    })
    match = frame.loc[frame["SeriesCode"].astype(str).eq(entry["code"])]
    if match.empty:
        raise KeyError(f"{entry['code']} not found in {filename}/{entry['sheet']}")
    label = str(match["Description"].iloc[0])
    if entry["label"].lower() not in label.lower():
        raise ValueError(f"Series-label mismatch for {entry['code']}: {label!r}")
    period_columns = [
        column for column in frame.columns if re.fullmatch(r"\d{4}Q[1-4]", str(column))
    ]
    values = pd.to_numeric(match.iloc[0][period_columns], errors="coerce") * entry["scale"]
    series = pd.Series(
        values.to_numpy(), index=pd.PeriodIndex(period_columns, freq="Q"), name=name
    ).dropna().sort_index()
    return series, label


official_real_series = {}
series_audit_rows = []
for series_name in REAL_SERIES_REGISTRY:
    series, label = load_bea_bulk_series(series_name)
    official_real_series[series_name] = series
    series_audit_rows.append({
        "series": series_name,
        "code": REAL_SERIES_REGISTRY[series_name]["code"],
        "label": label,
        "latest_quarter": str(series.index.max()),
    })
series_audit = pd.DataFrame(series_audit_rows)

# Ratio-splice the office price index before the data-center index begins.
dc_price = official_real_series["dc_structures_price"]
office_price = official_real_series["office_price"]
first_dc_quarter = dc_price.first_valid_index()
office_splice_ratio = dc_price.loc[first_dc_quarter] / office_price.loc[first_dc_quarter]
dc_price_spliced = dc_price.combine_first(office_price * office_splice_ratio).sort_index()

power_cooling_price = sum(
    official_real_series[name] * weight
    for name, weight in POWER_COOLING_PRICE_WEIGHTS.items()
)

category_prices = {
    "dc_structures": dc_price_spliced,
    "power_structures": official_real_series["power_structures_price"],
    "it_equipment": official_real_series["computers_price"],
    "power_cooling_equipment": power_cooling_price,
    "other_pfi": official_real_series["other_equipment_price"],
}
computer_import_price = official_real_series["computer_imports_price"]
display(series_audit)


In [ ]:
# Deflate the flows and estimate q/q annualized real GDP-growth contributions
real_asset_detail = asset_detail.loc[
    asset_detail["recorded_quarter"].between(MODEL_START, REAL_CONTRIBUTION_END)
].copy()
real_asset_detail["investment_price_index"] = [
    category_prices[category].get(quarter, np.nan)
    for category, quarter in zip(real_asset_detail["category"], real_asset_detail["recorded_quarter"])
]
real_asset_detail["import_price_index"] = real_asset_detail["investment_price_index"]
it_rows = real_asset_detail["category"].eq("it_equipment")
real_asset_detail.loc[it_rows, "import_price_index"] = [
    computer_import_price.get(quarter, np.nan)
    for quarter in real_asset_detail.loc[it_rows, "recorded_quarter"]
]

missing_investment_prices = int(real_asset_detail["investment_price_index"].isna().sum())
missing_import_prices = int(
    (real_asset_detail["associated_imports_bn"].gt(0) & real_asset_detail["import_price_index"].isna()).sum()
)
if missing_investment_prices or missing_import_prices:
    raise ValueError(
        f"Missing actual-quarter deflators: investment={missing_investment_prices}, imports={missing_import_prices}"
    )

real_asset_detail["gross_ai_investment_real_bn"] = (
    real_asset_detail["gross_ai_investment_bn"]
    / (real_asset_detail["investment_price_index"] / 100.0)
)
real_asset_detail["associated_imports_real_bn"] = (
    real_asset_detail["associated_imports_bn"]
    / (real_asset_detail["import_price_index"] / 100.0)
)

actual_index = pd.period_range(MODEL_START, REAL_CONTRIBUTION_END, freq="Q")
real_quarterly_flows = real_asset_detail.groupby("recorded_quarter").agg(
    gross_ai_investment_real_bn=("gross_ai_investment_real_bn", "sum"),
    associated_imports_real_bn=("associated_imports_real_bn", "sum"),
).reindex(actual_index, fill_value=0.0)
real_quarterly_flows.index.name = "quarter"
real_quarterly_flows["gross_ai_investment_real_saar_bn"] = (
    real_quarterly_flows["gross_ai_investment_real_bn"] * 4.0
)
real_quarterly_flows["associated_imports_real_saar_bn"] = (
    real_quarterly_flows["associated_imports_real_bn"] * 4.0
)
real_quarterly_flows["net_ai_gdp_flow_real_saar_bn"] = (
    real_quarterly_flows["gross_ai_investment_real_saar_bn"]
    - real_quarterly_flows["associated_imports_real_saar_bn"]
)

real_gdp = official_real_series["real_gdp"].reindex(actual_index)
if real_gdp.isna().any():
    raise ValueError("Real GDP is missing for one or more requested actual quarters.")

def qoq_annualized_growth(level):
    return ((level / level.shift(1)) ** QOQ_ANNUALIZATION_EXPONENT - 1.0) * 100.0

real_contributions = real_quarterly_flows.copy()
real_contributions["real_gdp_bn_saar"] = real_gdp
real_contributions["real_gdp_growth_qoq_saar_pct"] = qoq_annualized_growth(real_gdp)
gross_counterfactual = real_gdp - real_contributions["gross_ai_investment_real_saar_bn"]
net_counterfactual = (
    real_gdp
    - real_contributions["gross_ai_investment_real_saar_bn"]
    + real_contributions["associated_imports_real_saar_bn"]
)
real_contributions["real_gdp_growth_without_gross_ai_qoq_saar_pct"] = qoq_annualized_growth(gross_counterfactual)
real_contributions["real_gdp_growth_without_net_ai_qoq_saar_pct"] = qoq_annualized_growth(net_counterfactual)
real_contributions["real_gross_ai_contribution_pp"] = (
    real_contributions["real_gdp_growth_qoq_saar_pct"]
    - real_contributions["real_gdp_growth_without_gross_ai_qoq_saar_pct"]
)
real_contributions["real_net_ai_contribution_pp"] = (
    real_contributions["real_gdp_growth_qoq_saar_pct"]
    - real_contributions["real_gdp_growth_without_net_ai_qoq_saar_pct"]
)
real_contributions["real_import_contribution_pp"] = (
    real_contributions["real_net_ai_contribution_pp"]
    - real_contributions["real_gross_ai_contribution_pp"]
)
real_contributions["net_ai_share_of_real_gdp_growth_pct"] = (
    real_contributions["real_net_ai_contribution_pp"]
    / real_contributions["real_gdp_growth_qoq_saar_pct"] * 100.0
)

real_display_columns = [
    "real_gdp_growth_qoq_saar_pct",
    "real_gross_ai_contribution_pp",
    "real_import_contribution_pp",
    "real_net_ai_contribution_pp",
    "real_gdp_growth_without_net_ai_qoq_saar_pct",
    "net_ai_share_of_real_gdp_growth_pct",
]
display(real_contributions.loc[pd.Period("2024Q1", freq="Q"):REAL_CONTRIBUTION_END, real_display_columns])

latest_real_result = real_contributions.loc[REAL_CONTRIBUTION_END, real_display_columns]
print(f"{REAL_CONTRIBUTION_END} actual real GDP growth: {latest_real_result['real_gdp_growth_qoq_saar_pct']:.3f}% q/q annualized")
print(f"Estimated net AI contribution: {latest_real_result['real_net_ai_contribution_pp']:.3f} percentage points")
print(f"Estimated real GDP growth without net AI flow: {latest_real_result['real_gdp_growth_without_net_ai_qoq_saar_pct']:.3f}%")


In [ ]:
# Reconciliation checks and optional Excel export
same_quarter_2022_error = float(panel.loc[
    panel["year"].eq(2022) & panel["baseline_eligible"], "raw_excess_bn"
].abs().max())
imputed_status_count = int(panel["data_status"].astype(str).str.contains("imputed").sum())

us_source_total = model_panel["us_ai_capex_source_bn"].sum()
eligible_source_total = allocated.loc[
    ~allocated["category"].eq("land_excluded"), "source_amount_bn"
].sum()
land_total = land_excluded["source_amount_bn"].sum()
recognized_total = asset_detail["gross_ai_investment_bn"].sum()
net_identity_error = float((
    asset_detail["net_ai_gdp_flow_bn"]
    - asset_detail["gross_ai_investment_bn"]
    + asset_detail["associated_imports_bn"]
).abs().max())
real_net_identity_error = float((
    real_contributions["net_ai_gdp_flow_real_saar_bn"]
    - real_contributions["gross_ai_investment_real_saar_bn"]
    + real_contributions["associated_imports_real_saar_bn"]
).abs().max())
q2_2026_growth_rounding_difference = float(
    real_contributions.loc[pd.Period("2026Q2", freq="Q"), "real_gdp_growth_qoq_saar_pct"]
    - EXPECTED_2026Q2_REAL_GDP_GROWTH_SAAR
)

checks = pd.DataFrame({
    "check": [
        "category splits sum to 100%",
        "lag weights sum to 100%",
        "matched 2022 company-quarters have zero excess",
        "US allocation reconciles after land exclusion",
        "recognition timing conserves dollars through tail",
        "imports do not exceed investment",
        "net flow equals investment minus imports",
        "no CAPEX observation is imputed",
        "actual-quarter deflators are complete",
        "real net flow equals real investment minus real imports",
        "2026Q2 real GDP growth rounds to the 1.5% headline",
    ],
    "difference": [
        float((CATEGORY_SPLITS.sum(axis=1) - 1).abs().max()),
        max(abs(sum(weights) - 1) for weights in LAG_WEIGHTS.values()),
        same_quarter_2022_error,
        float(eligible_source_total + land_total - us_source_total),
        float(recognized_total - eligible_source_total),
        max(0.0, float((asset_detail["associated_imports_bn"] - asset_detail["gross_ai_investment_bn"]).max())),
        net_identity_error,
        float(imputed_status_count),
        float(missing_investment_prices + missing_import_prices),
        real_net_identity_error,
        q2_2026_growth_rounding_difference,
    ],
    "tolerance": [1e-12, 1e-12, 1e-12, 1e-8, 1e-8, 1e-12, 1e-12, 0.0, 0.0, 1e-12, 0.1],
})
checks["status"] = np.where(checks["difference"].abs() <= checks["tolerance"], "OK", "REVIEW")
display(checks)

assumption_rows = [
    ("LAST_ACTUAL_QUARTER", str(LAST_ACTUAL_QUARTER)),
    ("MODEL_START", str(MODEL_START)),
    ("ANALYSIS_END", str(ANALYSIS_END)),
    ("TAIL_END", str(TAIL_END)),
    ("CAPEX_INPUT", "Reported values only; no interpolation or imputation"),
    ("BASELINE", "Same company and same calendar quarter in 2022"),
    ("MISSING_2022_BASELINE", "Exclude observation from AI excess; do not assign zero"),
    ("SEASONALITY_ADJUSTMENT", False),
    ("AI_ATTRIBUTION", "100% of positive matched company-quarter excess"),
    ("US_SHARE_DEFAULT", US_SHARE_DEFAULT),
    ("US_SHARE_UTILITY", US_SHARE_UTILITY),
    ("EXPORTS_INCLUDED", INCLUDE_EXPORTS),
    ("GDP_GROWTH_CONTRIBUTIONS", "Actual quarters only through 2026Q2"),
    ("REAL_DEFLATION", "Asset-specific investment and import deflators"),
    ("REAL_CONTRIBUTION_METHOD", "q/q annualized additive chain-dollar counterfactual"),
    ("QOQ_ANNUALIZATION_EXPONENT", QOQ_ANNUALIZATION_EXPONENT),
    ("DATA_CENTER_PRICE_BACKCAST", "Office price index, ratio-spliced"),
    ("POWER_COOLING_DEFLATOR", "50% communication equipment / 50% electrical equipment"),
    ("NON_IT_IMPORT_DEFLATOR", "Matching domestic asset price index proxy"),
]
for profile, row in CATEGORY_SPLITS.iterrows():
    for category, value in row.items():
        assumption_rows.append((f"CATEGORY_SPLIT.{profile}.{category}", value))
for category, weights in LAG_WEIGHTS.items():
    assumption_rows.append((f"LAG_WEIGHTS.{category}", json.dumps(weights)))
for category, value in IMPORT_SHARES.items():
    assumption_rows.append((f"IMPORT_SHARE.{category}", value))
for company, profile in PROFILE_OVERRIDES.items():
    assumption_rows.append((f"PROFILE_OVERRIDE.{company}", profile))
assumptions_output = pd.DataFrame(assumption_rows, columns=["assumption", "value"])

if EXPORT_OUTPUTS:
    workbook_path = OUTPUT_DIR / OUTPUT_FILE
    company_output = model_panel[[
        "quarter", "Company_Name", "GICS_Industry_Group", "capital_profile", "data_status",
        "capex_reported", "baseline_flat_bn", "baseline_source", "baseline_eligible",
        "raw_excess_bn", "gross_ai_capex_bn", "us_share", "us_ai_capex_source_bn"
    ]].copy()
    company_output["quarter"] = company_output["quarter"].astype(str)
    quarterly_output = quarterly_results.reset_index()
    quarterly_output["quarter"] = quarterly_output["quarter"].astype(str)
    coverage_output = coverage_audit.loc[MODEL_START:ANALYSIS_END].reset_index()
    coverage_output["quarter"] = coverage_output["quarter"].astype(str)
    real_contributions_output = real_contributions.reset_index()
    real_contributions_output["quarter"] = real_contributions_output["quarter"].astype(str)

    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        annual_results.loc[2023:2028].reset_index().to_excel(writer, sheet_name="Annual Net Flow", index=False)
        quarterly_output.to_excel(writer, sheet_name="Quarterly Net Flow", index=False)
        real_contributions_output.to_excel(writer, sheet_name="Real GDP Contribution", index=False)
        company_output.to_excel(writer, sheet_name="Company Detail", index=False)
        industry_breakdown.to_excel(writer, sheet_name="Industry Breakdown", index=False)
        asset_breakdown.to_excel(writer, sheet_name="Asset Breakdown", index=False)
        coverage_output.to_excel(writer, sheet_name="Coverage Audit", index=False)
        baseline_audit.to_excel(writer, sheet_name="Baseline Audit", index=False)
        assumptions_output.to_excel(writer, sheet_name="Assumptions", index=False)
        series_audit.to_excel(writer, sheet_name="BEA Series", index=False)
        checks.to_excel(writer, sheet_name="Checks", index=False)
    print(f"Saved {workbook_path.relative_to(ROOT)}")

if checks["status"].eq("REVIEW").any():
    warnings.warn("One or more reconciliation checks require review.")
